# VDJBet YF Analysis (Rmd-aligned)

Python rewrite of `tmp/vdjbet_snippet.Rmd` with matching analysis outputs:

1. YF vs OLGA V-usage and V/J correction factors
2. LLW reference and adjusted mock generation
3. LLW Pgen histogram match against mock bins
4. LLW overlap per sample: matched clonotypes and duplicate_count
5. Cohen d, z-scores, empirical p-values, FDR
6. Red line (LLW) + mock boxplots and Cohen d heatmaps

In [15]:
import importlib.metadata as _meta
import sys as _sys
print(f"Python {_sys.version.split()[0]}")
for _pkg in ["mirpy-lib", "numpy", "pandas", "matplotlib", "scipy", "polars"]:
    try:
        print(f"  {_pkg}: {_meta.version(_pkg)}")
    except _meta.PackageNotFoundError:
        pass

import math
import sys
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from mir.comparative.vdjbet_workflow import (
    bh_fdr,
    build_real_control_analysis,
    build_synthetic_comparison,
    compute_bin_alignment_diagnostics,
    compute_olga_usage_adjustment,
    load_yfv_trb_samples,
    score_samples_dataframe,
)
from mir.common.parser import load_vdjdb_latest
from mir.utils.notebook_assets import ensure_airr_yfv19, notebook_large_assets_root

SEED = 42
N_MOCKS = 100
POOL_SIZE = 100_000
OLGA_USAGE_N = 1_000_000
# count_rearrangement (default, unweighted) or count_duplicates (weighted by duplicate_count)
USAGE_COUNT_MODE = "count_rearrangement"
USAGE_PSEUDOCOUNT = 1.0
YFV_CACHE_DIRNAME = "pkl_trb_repertoires"

ASSET_ROOT = notebook_large_assets_root(repo_root)
YFV_DIR = ensure_airr_yfv19(repo_root)

print(f"YFV_DIR = {YFV_DIR}")
print(f"ASSET_ROOT = {ASSET_ROOT}")
print(f"Usage mode = {USAGE_COUNT_MODE}, pseudocount = {USAGE_PSEUDOCOUNT}")
print(f"OLGA usage cache size = {OLGA_USAGE_N:,}")

# Publication-quality matplotlib style (Nature/Science aesthetics)
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "axes.grid": False,
})


/Users/mikesh/vcs/mirpy/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 45 files:   0%|          | 0/45 [00:00<?, ?it/s]

Fetching 45 files: 100%|██████████| 45/45 [00:00<00:00, 1907.00it/s]

YFV_DIR = /Users/mikesh/vcs/mirpy/notebooks/assets/large/airr_yfv19
ASSET_ROOT = /Users/mikesh/vcs/mirpy/notebooks/assets/large
Usage mode = count_rearrangement, pseudocount = 1.0
OLGA usage cache size = 1,000,000


## 1. Load LLWNGPMAV TRB reference from VDJdb

In [2]:
vdjdb_rep = load_vdjdb_latest(
    epitope="LLWNGPMAV",
    locus="TRB",
    species="HomoSapiens",
    mhc_a_contains="A*02",
)
print(f"Reference clonotypes: {vdjdb_rep.clonotype_count}")
print(f"Example: {vdjdb_rep.clonotypes[0].junction_aa}  {vdjdb_rep.clonotypes[0].v_gene}  {vdjdb_rep.clonotypes[0].j_gene}")

Downloading: https://github.com/antigenomics/vdjdb-db/releases/download/2025-12-29/vdjdb-2025-12-29.zip


LLWNGPMAV: 409 unique TRB clonotypes
Reference clonotypes: 409
Example: CAIQDAGASYEQYF  TRBV6-2*01  TRBJ2-7*01


## 2. Load YF samples

In [16]:
# Load YF TRB repertoires and build per-sample records for VDJBet.
warnings.filterwarnings("ignore", category=FutureWarning)

samples, yfv_gu = load_yfv_trb_samples(YFV_DIR)

print(f"Loaded TRB samples: {len(samples)}")
print(f"Total TRB clonotypes: {sum(s['repertoire'].clonotype_count for s in samples):,}")
print(f"Total TRB duplicates: {sum(s['repertoire'].duplicate_count for s in samples):,}")

Loaded TRB samples: 42
Total TRB clonotypes: 29,939,151
Total TRB duplicates: 56,732,204


## 3. OLGA usage and correction factors

Compute OLGA and YF usage frequencies, then derive correction factors.

Frequencies are computed in GeneUsage with configurable count mode and Laplace smoothing:

- mode count_rearrangement: count of unique clonotypes per key / total clonotypes
- mode count_duplicates: sum duplicate_count per key / total duplicate_count
- pseudocount 1 is added for both YF and OLGA sides

Formulas:

- factor_v = P_YF(V) / P_OLGA(V)
- factor_vj = P_YF(V,J) / P_OLGA(V,J)

These factors are used by PgenGeneUsageAdjustment.

In [ ]:
usage_result = compute_olga_usage_adjustment(
    yfv_gu,
    seed=SEED,
    n_jobs=8,
    olga_usage_n=OLGA_USAGE_N,
    count_mode=USAGE_COUNT_MODE,
    pseudocount=USAGE_PSEUDOCOUNT,
)

olga_model = usage_result.olga_model
olga_gu = usage_result.olga_usage
v_cmp = usage_result.v_cmp
vj_cmp = usage_result.vj_cmp
v_df = usage_result.v_df
vj_df = usage_result.vj_df

# Keep OLGA-based adjustment for the synthetic-null comparison section.
pgen_adj_olga = usage_result.pgen_adj_olga

print("Top V genes by |log2 factor|:")
display(v_df.assign(abs_log2=lambda d: d["log2_factor_v"].abs()).sort_values("abs_log2", ascending=False).head(15))
print("Top VJ pairs by |log2 factor|:")
display(vj_df.assign(abs_log2=lambda d: d["log2_factor_vj"].abs()).sort_values("abs_log2", ascending=False).head(15))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.clip(v_df["p_yf"].values, 1e-12, None)
y = np.clip(v_df["p_olga"].values, 1e-12, None)
sc = axes[0].scatter(x, y, c=v_df["log2_factor_v"].values, cmap="RdBu_r", s=40, alpha=0.9)
axes[0].plot([x.min(), x.max()], [x.min(), x.max()], color="#666666", linestyle=":", linewidth=1)
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel("P_YF(V)")
axes[0].set_ylabel("P_OLGA(V)")
axes[0].set_title(f"V usage frequencies: YF vs OLGA ({USAGE_COUNT_MODE}, n={OLGA_USAGE_N:,})")
axes[0].grid(alpha=0.2)
cb = plt.colorbar(sc, ax=axes[0], shrink=0.9)
cb.set_label("log2 correction factor")

top = v_df.assign(abs_log2=lambda d: d["log2_factor_v"].abs()).sort_values("abs_log2", ascending=False).head(20)
axes[1].barh(top["v_gene"], top["log2_factor_v"], color=["#c0392b" if z > 0 else "#2c3e50" for z in top["log2_factor_v"]])
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set_title("Top V correction factors (log2)")
axes[1].set_xlabel("log2(P_YF(V) / P_OLGA(V))")

plt.tight_layout()
plt.show()

print(f"Zero-frequency OLGA V genes in comparison table: {(v_df['p_olga'] == 0).sum()}")

Top V genes by |log2 factor|:


,v_gene,p_yf,p_olga,factor_v,log2_factor_v,abs_log2
42,TRBV7-4,0.000514,0.012746,0.040352,-4.631216,4.631216
34,TRBV6-3,0.000602,0.011164,0.053914,-4.213199,4.213199
38,TRBV6-8,0.000327,0.003023,0.108108,-3.209451,3.209451
39,TRBV6-9,0.000210,0.001793,0.116900,-3.096655,3.096655
0,TRBV10-1,0.003474,0.027009,0.128630,-2.958696,2.958696
9,TRBV13,0.004602,0.033960,0.135515,-2.883479,2.883479
35,TRBV6-4,0.004885,0.035608,0.137190,-2.865753,2.865753
5,TRBV11-3,0.004155,0.020005,0.207717,-2.267308,2.267308
6,TRBV12-3,0.071966,0.016579,4.340769,2.117950,2.117950
8,TRBV12-5,0.003422,0.012890,0.265444,-1.913523,1.913523


Top VJ pairs by |log2 factor|:


,v_gene,j_gene,p_yf,p_olga,factor_vj,log2_factor_vj,abs_log2
565,TRBV7-4,TRBJ1-2,0.000024,0.001247,0.019255,-5.698617,5.698617
575,TRBV7-4,TRBJ2-6,0.000006,0.000223,0.024879,-5.328923,5.328923
473,TRBV6-4,TRBJ1-3,0.000030,0.001142,0.026112,-5.259156,5.259156
71,TRBV11-3,TRBJ1-5,0.000054,0.002052,0.026178,-5.255528,5.255528
564,TRBV7-4,TRBJ1-1,0.000040,0.001467,0.027594,-5.179525,5.179525
474,TRBV6-4,TRBJ1-4,0.000049,0.001672,0.029087,-5.103489,5.103489
567,TRBV7-4,TRBJ1-4,0.000019,0.000585,0.032108,-4.960929,4.960929
462,TRBV6-3,TRBJ1-5,0.000039,0.001191,0.032917,-4.925014,4.925014
569,TRBV7-4,TRBJ1-6,0.000015,0.000445,0.033121,-4.916089,4.916089
568,TRBV7-4,TRBJ1-5,0.000049,0.001417,0.034742,-4.847183,4.847183


Zero-frequency OLGA V genes in comparison table: 0


/var/folders/w1/pqrcnlxn3ss93t6764fdgp1c0000gn/T/ipykernel_26801/2859193681.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Why real control?

Using a real human TRB control repertoire instead of a fully synthetic OLGA null corrects for biases that OLGA does not model: sequencing depth artifacts, PCR amplification skew, and repertoire sampling effects.

For this section we also compute V/J adjustment against the **real control** distribution (YF / real-control usage), not YF / OLGA. OLGA-based adjustment is kept for the synthetic-null comparison in section 9.

The histogram above should show that the real-control mock bin distribution (blue dashes) is close to the LLW reference (red). A large discrepancy would indicate that Pgen or gene-usage adjustment requires further tuning.

After the mock key sets are built here (lazy, first `.score()` call), each subsequent `.score()` call reuses them — only the query-repertoire counting differs per sample.

In [18]:
real_result = build_real_control_analysis(
    vdjdb_rep,
    yfv_gu,
    seed=SEED,
    count_mode=USAGE_COUNT_MODE,
    pseudocount=USAGE_PSEUDOCOUNT,
    pool_size=POOL_SIZE,
    n_mocks=N_MOCKS,
    n_jobs=8,
)

real_control_df = real_result.control_df
real_control_gu = real_result.control_usage
pgen_adj_real = real_result.pgen_adj_real
pool = real_result.pool
analysis = real_result.analysis
dt = real_result.elapsed_s
print(f"Pool built: n_generated={pool.n_generated:,}, bins={len(pool.bins)}, floor={pool.floor_bin}, ceil={pool.ceil_bin}, elapsed={dt:.1f}s")

diag = compute_bin_alignment_diagnostics(analysis)
all_bins = diag["all_bins"]
ref_vec = diag["ref_vec"]
mock_mat = diag["mock_mat"]
mock_mean = diag["mock_mean"]
mock_q10 = diag["mock_q10"]
mock_q90 = diag["mock_q90"]
max_abs_diff = diag["max_abs_diff"]
rmsd = diag["rmsd"]

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(all_bins, mock_q10, mock_q90, alpha=0.25, color="#1f77b4", label="Mock 10-90%", zorder=1)
ax.plot(all_bins, ref_vec, color="#d62728", marker="o", linewidth=2, label="LLW reference", zorder=2)
# Draw mock mean last so it remains visible even when overlapping the reference line.
ax.plot(all_bins, mock_mean, color="#1f77b4", linestyle="--", marker="s", markersize=4, linewidth=2.2, label="Mock mean", zorder=3)
ax.set_xlabel("log2 Pgen bin")
ax.set_ylabel("count")
ax.set_title("LLW reference vs real-control mock Pgen histogram")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print(f"Mock vs LLW: median max|Delta p(bin)|={np.median(max_abs_diff):.4f}, p95={np.percentile(max_abs_diff,95):.4f}")
print(f"Mock vs LLW: median RMSD={np.median(rmsd):.4f}, p95={np.percentile(rmsd,95):.4f}")

Pool built: n_generated=99,991, bins=47, floor=-64, ceil=-18, elapsed=117.9s


Mock vs LLW: median max|Delta p(bin)|=0.0000, p95=0.0000
Mock vs LLW: median RMSD=0.0000, p95=0.0000


/var/folders/w1/pqrcnlxn3ss93t6764fdgp1c0000gn/T/ipykernel_26801/2505957613.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. LLW overlap per sample: matched clonotypes and duplicate_count

Compute for each donor/day/replica:
- real LLW overlap
- mock distribution summary
- z-score and empirical p-value
- Cohen d
- FDR-adjusted p-values

In [19]:
df_res = score_samples_dataframe(analysis, samples, progress_every=10)

display(df_res[[
    "donor", "replica", "day",
    "matched_n_real", "matched_dc_real", "matched_n_fraction", "matched_dc_fraction",
    "matched_n_mock_mean", "matched_n_z", "matched_n_p_emp", "matched_n_cohen_d",
    "matched_dc_log2_real", "matched_dc_log2_mock_mean", "matched_dc_log2_z", "matched_dc_log2_p_emp", "matched_dc_log2_cohen_d",
]])

Processed 10/42 samples


Processed 20/42 samples


Processed 30/42 samples


Processed 40/42 samples


,donor,replica,day,matched_n_real,matched_dc_real,matched_n_fraction,matched_dc_fraction,matched_n_mock_mean,matched_n_z,matched_n_p_emp,matched_n_cohen_d,matched_dc_log2_real,matched_dc_log2_mock_mean,matched_dc_log2_z,matched_dc_log2_p_emp,matched_dc_log2_cohen_d
0,P1,F1,-1,11.0,55.0,0.000017,0.000049,22.77,-2.625422,1.000000,-2.625422,5.807355,5.917998,-0.228772,0.594059,-0.228772
1,P1,F1,0,13.0,48.0,0.000022,0.000057,21.07,-1.849660,0.970297,-1.849660,5.614710,5.527622,0.181657,0.485149,0.181657
2,P1,F1,7,21.0,100.0,0.000025,0.000062,28.36,-1.480386,0.960396,-1.480386,6.658211,6.475363,0.399458,0.366337,0.399458
3,P1,F1,15,28.0,364.0,0.000031,0.000206,26.69,0.297359,0.415842,0.297359,8.511753,6.404870,4.041158,0.009901,4.041158
4,P1,F1,45,18.0,432.0,0.000030,0.000452,22.15,-0.826377,0.792079,-0.826377,8.758223,5.729738,6.106591,0.009901,6.106591
5,P1,F2,0,11.0,45.0,0.000021,0.000058,20.04,-2.099272,0.990099,-2.099272,5.523562,5.365037,0.318021,0.405941,0.318021
6,P2,F1,-1,20.0,61.0,0.000019,0.000028,32.58,-2.196171,0.990099,-2.196171,5.954196,6.945716,-1.812018,0.970297,-1.812018
7,P2,F1,0,16.0,41.0,0.000026,0.000042,23.01,-1.478671,0.960396,-1.478671,5.392317,5.766441,-0.768835,0.792079,-0.768835
8,P2,F1,7,29.0,88.0,0.000026,0.000038,31.59,-0.506351,0.702970,-0.506351,6.475733,6.885887,-0.819281,0.792079,-0.819281
9,P2,F1,15,34.0,278.0,0.000029,0.000128,33.51,0.088703,0.534653,0.088703,8.124121,6.818966,2.843664,0.009901,2.843664


### Reading the overlap results

Four metrics are tracked per sample:

- **matched_n_real** — number of unique LLW CDR3+V+J clonotypes found in the sample.
- **matched_dc_real** — sum of `duplicate_count` over those matched clonotypes (read depth of the match).
- **matched_n_fraction** — `matched_n / n_total` (fraction of unique clonotypes in the sample that are LLW).
- **matched_dc_fraction** — `matched_dc / dc_total` (fraction of total read depth contributed by LLW matches).

Z-scores and Cohen d values are computed relative to the real-control mock null; positive values indicate enrichment above expectation.

In [7]:
plot_rows = []
for _, r in df_res.iterrows():
    for x in r["mock_n"]:
        plot_rows.append({
            "donor": r["donor"], "replica": r["replica"], "day": int(r["day"]),
            "metric": "matched_n", "kind": "mock", "value": float(x),
        })
    plot_rows.append({
        "donor": r["donor"], "replica": r["replica"], "day": int(r["day"]),
        "metric": "matched_n", "kind": "real", "value": float(r["matched_n_real"]),
    })
    # Sum of duplicate_count for matched clonotypes (raw read depth of LLW signal)
    for x in r["mock_dc_log2"]:
        plot_rows.append({
            "donor": r["donor"], "replica": r["replica"], "day": int(r["day"]),
            "metric": "matched_dc_log2", "kind": "mock", "value": float(x),
        })
    plot_rows.append({
        "donor": r["donor"], "replica": r["replica"], "day": int(r["day"]),
        "metric": "matched_dc_log2", "kind": "real", "value": float(r["matched_dc_log2_real"]),
    })
    # Fraction of unique clonotypes in sample that are LLW-matched
    plot_rows.append({
        "donor": r["donor"], "replica": r["replica"], "day": int(r["day"]),
        "metric": "matched_n_fraction", "kind": "real", "value": float(r["matched_n_fraction"]),
    })
    # Fraction of total read depth contributed by LLW matches
    plot_rows.append({
        "donor": r["donor"], "replica": r["replica"], "day": int(r["day"]),
        "metric": "matched_dc_fraction", "kind": "real", "value": float(r["matched_dc_fraction"]),
    })

plot_df = pd.DataFrame(plot_rows)
days_all = sorted(df_res["day"].unique().tolist())
donors = sorted(df_res["donor"].unique().tolist())
replicas = sorted(df_res["replica"].unique().tolist())


def draw_panel(metric, ylabel, title, with_mock=True):
    """Draw a grid of per-donor/replica line (real) + optional boxplot (mock) subplots."""
    fig, axes = plt.subplots(
        len(replicas), len(donors),
        figsize=(4.0 * len(donors), 3.2 * len(replicas)),
        squeeze=False,
    )
    fig.suptitle(title, fontsize=13, y=1.02)
    for ri, rep in enumerate(replicas):
        for di, donor in enumerate(donors):
            ax = axes[ri, di]
            sub = plot_df[
                (plot_df["metric"] == metric)
                & (plot_df["donor"] == donor)
                & (plot_df["replica"] == rep)
            ]
            if sub.empty:
                ax.set_visible(False)
                continue

            real = sub[sub["kind"] == "real"].sort_values("day")
            if with_mock:
                mock = sub[sub["kind"] == "mock"]
                box_data = [mock[mock["day"] == d]["value"].values for d in days_all]
                width = 2.5
                ax.boxplot(
                    box_data,
                    positions=days_all,
                    widths=width,
                    patch_artist=True,
                    boxprops=dict(facecolor="#d0e4f7", alpha=0.85),
                    medianprops=dict(color="#1f77b4", linewidth=1.6),
                    flierprops=dict(markersize=2),
                    manage_ticks=False,
                )
            ax.plot(real["day"], real["value"], "-o", color="#d62728", linewidth=2, markersize=5, zorder=5)
            ax.set_xticks(days_all)
            ax.set_xlabel("Day")
            ax.set_ylabel(ylabel)
            ax.set_title(f"{donor} {rep}", fontsize=9)
            ax.grid(alpha=0.2)

    plt.tight_layout()
    plt.show()


draw_panel("matched_n", "Matched clonotypes", "LLW matched clonotypes: real (line) vs mock (boxplots)")
draw_panel("matched_dc_log2", "log2(matched duplicate_count + 1)", "LLW matched duplicate_count: real (line) vs mock (boxplots)")
draw_panel("matched_n_fraction", "Fraction of total clonotypes", "LLW matched fraction (clonotypes)", with_mock=False)
draw_panel("matched_dc_fraction", "Fraction of total duplicate_count", "LLW matched fraction (read depth)", with_mock=False)

/var/folders/w1/pqrcnlxn3ss93t6764fdgp1c0000gn/T/ipykernel_26801/2108041686.py:82: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/w1/pqrcnlxn3ss93t6764fdgp1c0000gn/T/ipykernel_26801/2108041686.py:82: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Cohen d heatmaps

In [8]:
def heatmap_cohen(value_col, p_adj_col, title, cbar_label, cmap="RdBu_r", vlim=None):
    work = df_res.copy()
    work["sample"] = work["donor"] + " " + work["replica"]
    pv = work.pivot_table(index="sample", columns="day", values=value_col, aggfunc="first")
    pp = work.pivot_table(index="sample", columns="day", values=p_adj_col, aggfunc="first")

    mat = pv.values.astype(float)
    if vlim is None:
        vmax = max(1.0, np.nanpercentile(np.abs(mat), 95))
        vmin = -vmax
    else:
        vmin, vmax = vlim

    fig, ax = plt.subplots(figsize=(8, max(3.5, 0.6 * pv.shape[0])))
    im = ax.imshow(mat, cmap=cmap, aspect="auto", vmin=vmin, vmax=vmax, interpolation="nearest")
    cb = plt.colorbar(im, ax=ax, shrink=0.85)
    cb.set_label(cbar_label)

    for r in range(pv.shape[0]):
        for c in range(pv.shape[1]):
            p = pp.values[r, c]
            d = pv.values[r, c]
            if pd.notna(p) and pd.notna(d) and (float(p) < 0.10) and (float(d) > 0):
                ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, fill=False, edgecolor="black", linewidth=2))

    ax.set_xticks(range(len(pv.columns)))
    ax.set_xticklabels([f"Day {d}" for d in pv.columns], rotation=45, ha="right")
    ax.set_yticks(range(len(pv.index)))
    ax.set_yticklabels(pv.index)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

heatmap_cohen(
    value_col="matched_n_cohen_d",
    p_adj_col="matched_n_p_adj",
    title="Matched clonotypes Cohen d (LLW vs mock)",
    cbar_label="Cohen d (matched clonotypes)",
    cmap="RdBu_r",
)

heatmap_cohen(
    value_col="matched_dc_log2_cohen_d",
    p_adj_col="matched_dc_log2_p_adj",
    title="Matched duplicate_count Cohen d (log2, LLW vs mock)",
    cbar_label="Cohen d (log2 matched duplicate_count)",
    cmap="Reds",
    vlim=(0, max(1.0, np.nanpercentile(df_res["matched_dc_log2_cohen_d"].values, 95))),
)

/var/folders/w1/pqrcnlxn3ss93t6764fdgp1c0000gn/T/ipykernel_26801/2661021277.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/w1/pqrcnlxn3ss93t6764fdgp1c0000gn/T/ipykernel_26801/2661021277.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Final summary tables

In [9]:
summary_cols = [
    "donor", "replica", "day",
    "n_total", "dc_total",
    "matched_n_real", "matched_dc_real", "matched_n_fraction", "matched_dc_fraction",
    "matched_n_mock_mean", "matched_n_mock_sd", "matched_n_cohen_d", "matched_n_z", "matched_n_p_emp", "matched_n_p_adj",
    "matched_dc_log2_real", "matched_dc_log2_mock_mean", "matched_dc_log2_mock_sd",
    "matched_dc_log2_cohen_d", "matched_dc_log2_z", "matched_dc_log2_p_emp", "matched_dc_log2_p_adj",
]
summary = df_res[summary_cols].copy()
for col in [
    "matched_n_real", "matched_dc_real",
    "matched_n_mock_mean", "matched_n_mock_sd", "matched_n_cohen_d", "matched_n_z",
    "matched_dc_log2_real", "matched_dc_log2_mock_mean", "matched_dc_log2_mock_sd", "matched_dc_log2_cohen_d", "matched_dc_log2_z",
]:
    summary[col] = summary[col].astype(float).round(3)
for col in ["matched_n_fraction", "matched_dc_fraction"]:
    summary[col] = summary[col].map(lambda x: f"{x:.5f}")
for col in ["matched_n_p_emp", "matched_n_p_adj", "matched_dc_log2_p_emp", "matched_dc_log2_p_adj"]:
    summary[col] = summary[col].map(lambda x: f"{x:.4f}")

display(summary.sort_values(["donor", "replica", "day"]).reset_index(drop=True))

print("Top positive matched clonotype effects by Cohen d:")
display(summary.sort_values("matched_n_cohen_d", ascending=False).head(12))

print("Top positive matched duplicate_count effects by Cohen d:")
display(summary.sort_values("matched_dc_log2_cohen_d", ascending=False).head(12))

,donor,replica,day,n_total,dc_total,matched_n_real,matched_dc_real,matched_n_fraction,matched_dc_fraction,matched_n_mock_mean,...,matched_n_z,matched_n_p_emp,matched_n_p_adj,matched_dc_log2_real,matched_dc_log2_mock_mean,matched_dc_log2_mock_sd,matched_dc_log2_cohen_d,matched_dc_log2_z,matched_dc_log2_p_emp,matched_dc_log2_p_adj
0,P1,F1,-1,640262,1113509,11.0,55.0,0.00002,0.00005,22.77,...,-2.625,1.0000,1.0000,5.807,5.918,0.484,-0.229,-0.229,0.5941,1.0000
1,P1,F1,0,590253,848323,13.0,48.0,0.00002,0.00006,21.07,...,-1.850,0.9703,1.0000,5.615,5.528,0.479,0.182,0.182,0.4851,1.0000
2,P1,F1,7,852394,1623860,21.0,100.0,0.00002,0.00006,28.36,...,-1.480,0.9604,1.0000,6.658,6.475,0.458,0.399,0.399,0.3663,0.9051
3,P1,F1,15,901914,1764263,28.0,364.0,0.00003,0.00021,26.69,...,0.297,0.4158,1.0000,8.512,6.405,0.521,4.041,4.041,0.0099,0.1040
4,P1,F1,45,598677,956614,18.0,432.0,0.00003,0.00045,22.15,...,-0.826,0.7921,1.0000,8.758,5.730,0.496,6.107,6.107,0.0099,0.1040
5,P1,F2,0,534632,772165,11.0,45.0,0.00002,0.00006,20.04,...,-2.099,0.9901,1.0000,5.524,5.365,0.498,0.318,0.318,0.4059,0.9472
6,P2,F1,-1,1056617,2177498,20.0,61.0,0.00002,0.00003,32.58,...,-2.196,0.9901,1.0000,5.954,6.946,0.547,-1.812,-1.812,0.9703,1.0000
7,P2,F1,0,618627,974518,16.0,41.0,0.00003,0.00004,23.01,...,-1.479,0.9604,1.0000,5.392,5.766,0.487,-0.769,-0.769,0.7921,1.0000
8,P2,F1,7,1106211,2324263,29.0,88.0,0.00003,0.00004,31.59,...,-0.506,0.7030,1.0000,6.476,6.886,0.501,-0.819,-0.819,0.7921,1.0000
9,P2,F1,15,1175459,2165475,34.0,278.0,0.00003,0.00013,33.51,...,0.089,0.5347,1.0000,8.124,6.819,0.459,2.844,2.844,0.0099,0.1040


Top positive matched clonotype effects by Cohen d:


,donor,replica,day,n_total,dc_total,matched_n_real,matched_dc_real,matched_n_fraction,matched_dc_fraction,matched_n_mock_mean,...,matched_n_z,matched_n_p_emp,matched_n_p_adj,matched_dc_log2_real,matched_dc_log2_mock_mean,matched_dc_log2_mock_sd,matched_dc_log2_cohen_d,matched_dc_log2_z,matched_dc_log2_p_emp,matched_dc_log2_p_adj
28,S1,F1,15,669972,1069012,42.0,814.0,0.00006,0.00076,26.88,...,3.137,0.0099,0.4158,9.671,6.031,0.590,6.168,6.168,0.0099,0.1040
22,Q2,F1,15,607413,1449423,32.0,214.0,0.00005,0.00015,23.37,...,2.050,0.0495,1.0000,7.748,6.263,0.822,1.808,1.808,0.0297,0.1559
35,S2,F1,15,821039,1611965,39.0,471.0,0.00005,0.00029,31.91,...,1.487,0.0990,1.0000,8.883,6.852,0.653,3.112,3.112,0.0198,0.1188
29,S1,F1,45,572955,972623,28.0,368.0,0.00005,0.00038,22.94,...,1.253,0.1188,1.0000,8.527,5.858,0.775,3.444,3.444,0.0396,0.1848
16,Q1,F1,15,447584,843549,23.0,237.0,0.00005,0.00028,18.12,...,1.207,0.1980,1.0000,7.895,5.297,0.675,3.847,3.847,0.0198,0.1188
40,S2,F2,15,786155,1544994,34.0,424.0,0.00004,0.00027,29.69,...,0.801,0.2475,1.0000,8.731,6.738,0.675,2.952,2.952,0.0198,0.1188
3,P1,F1,15,901914,1764263,28.0,364.0,0.00003,0.00021,26.69,...,0.297,0.4158,1.0000,8.512,6.405,0.521,4.041,4.041,0.0099,0.1040
27,S1,F1,7,431736,652745,21.0,32.0,0.00005,0.00005,20.05,...,0.246,0.4455,1.0000,5.044,5.475,0.646,-0.667,-0.667,0.7921,1.0000
17,Q1,F1,45,442934,833618,19.0,85.0,0.00004,0.00010,18.38,...,0.164,0.4752,1.0000,6.426,5.359,0.628,1.701,1.701,0.0693,0.2426
9,P2,F1,15,1175459,2165475,34.0,278.0,0.00003,0.00013,33.51,...,0.089,0.5347,1.0000,8.124,6.819,0.459,2.844,2.844,0.0099,0.1040


Top positive matched duplicate_count effects by Cohen d:


,donor,replica,day,n_total,dc_total,matched_n_real,matched_dc_real,matched_n_fraction,matched_dc_fraction,matched_n_mock_mean,...,matched_n_z,matched_n_p_emp,matched_n_p_adj,matched_dc_log2_real,matched_dc_log2_mock_mean,matched_dc_log2_mock_sd,matched_dc_log2_cohen_d,matched_dc_log2_z,matched_dc_log2_p_emp,matched_dc_log2_p_adj
28,S1,F1,15,669972,1069012,42.0,814.0,0.00006,0.00076,26.88,...,3.137,0.0099,0.4158,9.671,6.031,0.590,6.168,6.168,0.0099,0.1040
4,P1,F1,45,598677,956614,18.0,432.0,0.00003,0.00045,22.15,...,-0.826,0.7921,1.0000,8.758,5.730,0.496,6.107,6.107,0.0099,0.1040
3,P1,F1,15,901914,1764263,28.0,364.0,0.00003,0.00021,26.69,...,0.297,0.4158,1.0000,8.512,6.405,0.521,4.041,4.041,0.0099,0.1040
16,Q1,F1,15,447584,843549,23.0,237.0,0.00005,0.00028,18.12,...,1.207,0.1980,1.0000,7.895,5.297,0.675,3.847,3.847,0.0198,0.1188
29,S1,F1,45,572955,972623,28.0,368.0,0.00005,0.00038,22.94,...,1.253,0.1188,1.0000,8.527,5.858,0.775,3.444,3.444,0.0396,0.1848
35,S2,F1,15,821039,1611965,39.0,471.0,0.00005,0.00029,31.91,...,1.487,0.0990,1.0000,8.883,6.852,0.653,3.112,3.112,0.0198,0.1188
40,S2,F2,15,786155,1544994,34.0,424.0,0.00004,0.00027,29.69,...,0.801,0.2475,1.0000,8.731,6.738,0.675,2.952,2.952,0.0198,0.1188
9,P2,F1,15,1175459,2165475,34.0,278.0,0.00003,0.00013,33.51,...,0.089,0.5347,1.0000,8.124,6.819,0.459,2.844,2.844,0.0099,0.1040
22,Q2,F1,15,607413,1449423,32.0,214.0,0.00005,0.00015,23.37,...,2.050,0.0495,1.0000,7.748,6.263,0.822,1.808,1.808,0.0297,0.1559
13,Q1,F1,-1,393851,930778,13.0,101.0,0.00003,0.00011,16.78,...,-0.985,0.8812,1.0000,6.672,5.315,0.752,1.805,1.805,0.0594,0.2268


Notebook output coverage checklist:

- V usage YF vs OLGA and correction factors
- LLW reference vs mock Pgen histogram alignment
- Matched clonotypes and duplicate_count per sample
- Cohen d, z-scores, empirical p-values, FDR
- Line + boxplot dynamics and Cohen d heatmaps

In [10]:
print("Done: notebook rewritten to Rmd-aligned workflow.")
print(f"Samples: {len(samples)}, mocks: {N_MOCKS}, pool size: {POOL_SIZE:,}")
print("Use the summary table above for export/reporting.")

Done: notebook rewritten to Rmd-aligned workflow.
Samples: 42, mocks: 100, pool size: 100,000
Use the summary table above for export/reporting.


## 9. Synthetic mock comparison and scale-factor calibration

Sections 1–8 use a **real human TRB control pool** to drive the mock null distribution.
This section builds a complementary **synthetic OLGA pool** and rescores all samples under
that null, then computes a scale factor $X$ aligning the two mean mock overlaps:

$$X = \frac{\bar{m}_{\text{real}}}{\bar{m}_{\text{synth}}}$$

where $\bar{m}$ denotes the mean number of mocked matches across all samples.
Three mock curves are overlaid per day:

- **mock real** — null from the real-control pool (sections 1–8).
- **mock synthetic** — null from OLGA synthetic sequences.
- **mock synthetic × X** — synthetic rescaled to match the real-pool mean.

The scale factor quantifies how much OLGA under- or over-estimates real-sequence
background overlap density.  A value $X > 1$ means the real control finds more
background matches than OLGA predicts (typical due to empirical sequence clustering).

In [11]:
import importlib
import mir.common.control as _ctrl_mod
import mir.comparative.vdjbet as _vdjbet_mod
importlib.reload(_ctrl_mod)
importlib.reload(_vdjbet_mod)
from mir.comparative.vdjbet import PgenBinPool, VDJBetOverlapAnalysis
print('Reloaded mir.common.control and mir.comparative.vdjbet from source')

Reloaded mir.common.control and mir.comparative.vdjbet from source


In [20]:
df_res_real = df_res.copy()  # real-control null from sections 1-8

(
    pool_synth,
    analysis_synth,
    df_res_synth,
    X_scale,
    df_res_synth_scaled,
    ) = build_synthetic_comparison(
        vdjdb_rep,
        samples,
        pgen_adj_olga=pgen_adj_olga,
        pool_size=POOL_SIZE,
        n_mocks=N_MOCKS,
        n_jobs=8,
        seed=SEED,
        df_res_real=df_res_real,
    )

print(f"Synthetic pool records used: {pool_synth.n_generated:,}")
print(f"Synthetic mock mean overlap: {df_res_synth['matched_n_mock_mean'].mean():.4f}")
print(f"Real-control mock mean overlap: {df_res_real['matched_n_mock_mean'].mean():.4f}")
print(f"Scale factor X (real/synthetic): {X_scale:.4f}")

Processed 10/42 samples


Processed 20/42 samples


Processed 30/42 samples


Processed 40/42 samples


Synthetic pool records used: 100,000
Synthetic mock mean overlap: 8.7114
Real-control mock mean overlap: 25.3510
Scale factor X (real/synthetic): 2.9101


In [13]:
# Significant hits under real-control null (from sections 1-8).
sig_n_real = df_res_real[
    (df_res_real["matched_n_p_adj"] < 0.10) & (df_res_real["matched_n_cohen_d"] > 0)
]
sig_dc_real = df_res_real[
    (df_res_real["matched_dc_log2_p_adj"] < 0.10) & (df_res_real["matched_dc_log2_cohen_d"] > 0)
]


def _as_triplet_set(df):
    return {(r.donor, r.replica, int(r.day)) for r in df[["donor", "replica", "day"]].itertuples(index=False)}


def _as_label_list(df):
    return [
        f"{r.donor} {r.replica} day {int(r.day)}"
        for r in df[["donor", "replica", "day"]].itertuples(index=False)
    ]


# Expected significant sets based on the YFV19 Rmd analysis (day 15 peak responders).
expected_sig_n = {("S2", "F1", 15), ("S1", "F1", 15), ("Q2", "F1", 15), ("Q1", "F1", 15)}
expected_sig_dc = {
    ("S2", "F1", 15), ("S2", "F2", 15),
    ("S1", "F1", 15), ("S1", "F1", 45),
    ("Q1", "F1", 15),
    ("P2", "F1", 15),
    ("P1", "F1", 15), ("P1", "F1", 45),
}

set_sig_n = _as_triplet_set(sig_n_real)
set_sig_dc = _as_triplet_set(sig_dc_real)

print("Observed significant matched clonotypes (FDR<0.10, d>0):")
print(_as_label_list(sig_n_real))
print("Observed significant matched duplicate_count log2 (FDR<0.10, d>0):")
print(_as_label_list(sig_dc_real))

print("\nExpectation check: matched clonotypes")
print("match:", set_sig_n == expected_sig_n)
print("missing:", sorted(expected_sig_n - set_sig_n))
print("extra:", sorted(set_sig_n - expected_sig_n))

print("\nExpectation check: matched duplicate_count")
print("match:", set_sig_dc == expected_sig_dc)
print("missing:", sorted(expected_sig_dc - set_sig_dc))
print("extra:", sorted(set_sig_dc - expected_sig_dc))

print(f"\nScale factor X (real/synthetic): {X_scale:.3f}")
print("Observation 1: Real-control null captures empirical sequence clustering that OLGA misses.")
print(f"Observation 2: X = {X_scale:.3f} — synthetic mocks undercount overlap by this factor.")
print("Observation 3: Significant calls are exact junction_aa + V + J matching only.")
print("Observation 4: Compare listed observed sets with target day/sample expectations via missing/extra diagnostics.")

Observed significant matched clonotypes (FDR<0.10, d>0):
[]
Observed significant matched duplicate_count log2 (FDR<0.10, d>0):
[]

Expectation check: matched clonotypes
match: False
missing: [('Q1', 'F1', 15), ('Q2', 'F1', 15), ('S1', 'F1', 15), ('S2', 'F1', 15)]
extra: []

Expectation check: matched duplicate_count
match: False
missing: [('P1', 'F1', 15), ('P1', 'F1', 45), ('P2', 'F1', 15), ('Q1', 'F1', 15), ('S1', 'F1', 15), ('S1', 'F1', 45), ('S2', 'F1', 15), ('S2', 'F2', 15)]
extra: []

Scale factor X (real/synthetic): 2.910
Observation 1: Real-control null captures empirical sequence clustering that OLGA misses.
Observation 2: X = 2.910 — synthetic mocks undercount overlap by this factor.
Observation 3: Significant calls are exact junction_aa + V + J matching only.
Observation 4: Compare listed observed sets with target day/sample expectations via missing/extra diagnostics.


In [22]:
days_plot = sorted(df_res_real["day"].unique().tolist())


def _collect_mock_by_day(df_in, value_col="mock_n", transform=None):
    """Aggregate mock draws per day from a scored DataFrame."""
    out = {d: [] for d in days_plot}
    for _, row in df_in.iterrows():
        d = int(row["day"])
        vals = [float(x) for x in row[value_col]]
        if transform is not None:
            vals = [float(transform(v)) for v in vals]
        out[d].extend(vals)
    return out


real_mock_by_day = _collect_mock_by_day(df_res_real, "mock_n")
syn_mock_by_day = _collect_mock_by_day(df_res_synth, "mock_n")
syn_scaled_mock_by_day = _collect_mock_by_day(df_res_synth_scaled, "mock_n")

# Duplicate-count mock draws are stored as log2(mock_dc + 1). Convert back to raw counts.
_dc_from_log2 = lambda v: (2.0 ** float(v)) - 1.0
real_mock_dc_by_day = _collect_mock_by_day(df_res_real, "mock_dc_log2", transform=_dc_from_log2)
syn_mock_dc_by_day = _collect_mock_by_day(df_res_synth, "mock_dc_log2", transform=_dc_from_log2)
syn_scaled_mock_dc_by_day = _collect_mock_by_day(
    df_res_synth_scaled, "mock_dc_log2", transform=lambda v: _dc_from_log2(v) * X_scale
)

# Real-data mean overlap per day (from sections 1-8, real-control null).
real_mean_by_day = (
    df_res_real.groupby("day", as_index=False)["matched_n_real"].mean().sort_values("day")
)
real_dc_by_day = (
    df_res_real.groupby("day", as_index=False)["matched_dc_real"].mean().sort_values("day")
)
real_frac_n_by_day = (
    df_res_real.groupby("day", as_index=False)["matched_n_fraction"].mean().sort_values("day")
)
real_frac_dc_by_day = (
    df_res_real.groupby("day", as_index=False)["matched_dc_fraction"].mean().sort_values("day")
)

width = 1.5
offsets = [-1.8, 0.0, 1.8]
mock_specs = [
    ("mock real", real_mock_by_day, "#f6b26b", offsets[0]),
    ("mock synthetic", syn_mock_by_day, "#87b4e3", offsets[1]),
    (f"mock synthetic × {X_scale:.2f}", syn_scaled_mock_by_day, "#a4d4ae", offsets[2]),
]
mock_dc_specs = [
    ("mock real", real_mock_dc_by_day, "#f6b26b", offsets[0]),
    ("mock synthetic", syn_mock_dc_by_day, "#87b4e3", offsets[1]),
    (f"mock synthetic × {X_scale:.2f}", syn_scaled_mock_dc_by_day, "#a4d4ae", offsets[2]),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("LLW overlap: real (line) vs mock null distributions (boxplots)", fontsize=14)

# ── Panel A: matched clonotype count ──────────────────────────────────────────
ax = axes[0, 0]
for label, by_day, color, off in mock_specs:
    data = [by_day[d] if len(by_day[d]) else [np.nan] for d in days_plot]
    pos = [d + off for d in days_plot]
    ax.boxplot(
        data, positions=pos, widths=width, patch_artist=True, manage_ticks=False,
        boxprops=dict(facecolor=color, alpha=0.55),
        medianprops=dict(color="#2f2f2f", linewidth=1.6),
        flierprops=dict(markersize=1.5),
    )
    ax.plot([], [], color=color, linewidth=8, alpha=0.55, label=label)
ax.plot(real_mean_by_day["day"], real_mean_by_day["matched_n_real"],
        "-o", color="#d62728", linewidth=2.2, markersize=5, label="real mean")
ax.set_xticks(days_plot)
ax.set_xlabel("Day")
ax.set_ylabel("Matched clonotypes")
ax.set_title("(A) Matched clonotype count")
ax.grid(alpha=0.2)
ax.legend(loc="upper right", fontsize=8)

# ── Panel B: sum of duplicate_count in matched clonotypes ─────────────────────
ax = axes[0, 1]
for label, by_day, color, off in mock_dc_specs:
    data_raw = [by_day[d] if len(by_day[d]) else [np.nan] for d in days_plot]
    # Use log2(x + 1) scale for duplicate-count panel.
    data = [np.log2(np.asarray(vals, dtype=float) + 1.0) for vals in data_raw]
    pos = [d + off for d in days_plot]
    ax.boxplot(
        data, positions=pos, widths=width, patch_artist=True, manage_ticks=False,
        boxprops=dict(facecolor=color, alpha=0.55),
        medianprops=dict(color="#2f2f2f", linewidth=1.6),
        flierprops=dict(markersize=1.5),
    )
    ax.plot([], [], color=color, linewidth=8, alpha=0.55, label=label)
ax.plot(
    real_dc_by_day["day"],
    np.log2(real_dc_by_day["matched_dc_real"].astype(float).values + 1.0),
    "-o", color="#d62728", linewidth=2.2, markersize=5, label="real mean"
)
ax.set_xticks(days_plot)
ax.set_xlabel("Day")
ax.set_ylabel("log2(sum of duplicate_count (matched) + 1)")
ax.set_title("(B) Sum of duplicate_count in matched clonotypes")
ax.grid(alpha=0.2)
ax.legend(loc="upper right", fontsize=8)

# ── Panel C: fraction of clonotypes (matched / total) ─────────────────────────
ax = axes[1, 0]
ax.plot(real_frac_n_by_day["day"], real_frac_n_by_day["matched_n_fraction"],
        "-o", color="#d62728", linewidth=2.2, markersize=5, label="real mean")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.4%}"))
ax.set_xticks(days_plot)
ax.set_xlabel("Day")
ax.set_ylabel("Fraction of total clonotypes")
ax.set_title("(C) Matched clonotype fraction (matched / total clonotypes)")
ax.grid(alpha=0.2)
ax.legend()

# ── Panel D: fraction of duplicate_count (matched / total) ────────────────────
ax = axes[1, 1]
ax.plot(real_frac_dc_by_day["day"], real_frac_dc_by_day["matched_dc_fraction"],
        "-o", color="#d62728", linewidth=2.2, markersize=5, label="real mean")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.4%}"))
ax.set_xticks(days_plot)
ax.set_xlabel("Day")
ax.set_ylabel("Fraction of total duplicate_count")
ax.set_title("(D) Matched read-depth fraction (matched / total duplicate_count)")
ax.grid(alpha=0.2)
ax.legend()

plt.tight_layout()
plt.show()

/var/folders/w1/pqrcnlxn3ss93t6764fdgp1c0000gn/T/ipykernel_26801/2590439297.py:130: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
